# Pydantic 인자와 반환 스키마

허용 값과 개수 범위를 실행 전에 검사합니다.

## 준비

## 코드를 읽기 전에

Pydantic 모델은 실행 전에 입력 값을 확인하는 클래스입니다. `Literal`은 허용 문자열을 제한하고, `Field(ge=1, le=10)`은 `limit`의 최소·최대 범위를 정합니다. `args_schema=OrderQuery`를 `@tool`에 넘기면 이 규칙이 도구의 JSON Schema가 되고, `invoke()`는 함수 본문을 실행하기 전에 값을 검증합니다.

반환 쪽의 `TypeAdapter(list[OrderRecord])`는 함수가 만든 목록의 각 사전을 다시 검사합니다. 입력 검증은 호출 모양을, 반환 검증은 결과 모양을 확인합니다. 주문이 실제 고객의 것인지 같은 업무 사실은 별도의 데이터 조회와 권한 검사가 필요합니다.

In [1]:
# 상위 폴더에서 환경 파일과 공개 주문 CSV를 찾습니다.
from pathlib import Path
from dotenv import load_dotenv

def find_up(name: str) -> Path | None:
    current = Path.cwd().resolve()
    for folder in (current, *current.parents):
        candidate = folder / name
        if candidate.exists():
            return candidate
    return None

load_dotenv(find_up(".env"))
data_path = find_up("data/orders.csv")
assert data_path is not None, "공개 묶음의 챕터 폴더 또는 notebooks 폴더에서 실행합니다."
print("Python 실습 준비 완료")


Python 실습 준비 완료


### 다음 셀을 읽는 방법

다음 셀은 Ch 4의 `fetch_orders()`를 이 노트북에서 독립 실행할 수 있도록 준비합니다. CSV 경로와 선택 상태를 입력받고 사전 목록을 반환하며, 뒤에서 Pydantic 입력·반환 검증의 대상 함수가 됩니다.


In [2]:
# Ch 4의 조회 함수를 준비합니다.
from pathlib import Path
from typing import Any

import pandas as pd


def fetch_orders(csv_path: str | Path, *, status: str | None = None) -> list[dict[str, Any]]:
    """CSV 주문을 읽고 선택한 상태의 행을 사전 목록으로 반환합니다."""
    orders = pd.read_csv(csv_path, dtype={"order_id": "string", "customer_id": "string"})
    if status is not None:
        orders = orders.loc[orders["status"] == status]
    return orders.to_dict(orient="records")


## 스키마와 정상 실행

In [3]:
# 허용 상태와 조회 개수 범위를 입력 스키마로 제한합니다.
from typing import Literal
from pydantic import BaseModel, Field
from langchain.tools import tool
class OrderQuery(BaseModel):
    status: Literal["paid", "pending", "cancelled"] = Field(description="조회할 주문 상태")
    limit: int = Field(default=3, ge=1, le=10, description="반환할 최대 주문 수")
@tool(args_schema=OrderQuery)
def search_orders(status: str, limit: int = 3) -> list[dict]:
    """주문 상태로 주문을 조회하고 최대 limit개를 반환합니다."""
    return fetch_orders(data_path, status=status)[:limit]
print(search_orders.args_schema.model_json_schema())
print(search_orders.invoke({"status": "paid", "limit": 2}))


{'properties': {'status': {'description': '조회할 주문 상태', 'enum': ['paid', 'pending', 'cancelled'], 'title': 'Status', 'type': 'string'}, 'limit': {'default': 3, 'description': '반환할 최대 주문 수', 'maximum': 10, 'minimum': 1, 'title': 'Limit', 'type': 'integer'}}, 'required': ['status'], 'title': 'OrderQuery', 'type': 'object'}
[{'order_id': 'O101', 'customer_id': 'C001', 'status': 'paid', 'amount': 32000}, {'order_id': 'O103', 'customer_id': 'C001', 'status': 'paid', 'amount': 27000}]


## 반환 형식

In [4]:
# 반환 행을 결과 모델로 검증하고 필드 이름을 확인합니다.
from pydantic import TypeAdapter
class OrderRecord(BaseModel):
    order_id: str
    customer_id: str
    status: Literal["paid", "pending", "cancelled"]
    amount: float = Field(ge=0)
value = search_orders.invoke({"status": "paid", "limit": 1})
validated = TypeAdapter(list[OrderRecord]).validate_python(value)
assert len(validated) == 1 and validated[0].order_id == "O101"
print({"type": type(value).__name__, "fields": sorted(value[0]),
       "validated": [row.model_dump() for row in validated]})


{'type': 'list', 'fields': ['amount', 'customer_id', 'order_id', 'status'], 'validated': [{'order_id': 'O101', 'customer_id': 'C001', 'status': 'paid', 'amount': 32000.0}]}


## 실패 재현

In [5]:
# 잘못된 상태와 조회 개수의 오류 위치를 확인합니다.
from pydantic import ValidationError
for invalid in [{"status": "shipped", "limit": 1}, {"status": "paid", "limit": 0}]:
    try:
        search_orders.invoke(invalid)
    except ValidationError as error:
        item = error.errors()[0]
        print({"input": invalid, "type": item["type"], "loc": item["loc"]})


{'input': {'status': 'shipped', 'limit': 1}, 'type': 'literal_error', 'loc': ('status',)}
{'input': {'status': 'paid', 'limit': 0}, 'type': 'greater_than_equal', 'loc': ('limit',)}


## 수정

In [6]:
# 허용된 상태와 개수로 바꾸어 실제 주문을 확인합니다.
result = search_orders.invoke({"status": "pending", "limit": 1})
assert len(result) == 1 and result[0]["order_id"] == "O102"
print(result)


[{'order_id': 'O102', 'customer_id': 'C002', 'status': 'pending', 'amount': 18000}]


## 다른 상황에 적용하기

호텔 검색 도구에서 인원은 1 이상, 객실 등급은 허용 목록으로 제한할 수 있습니다. 하지만 체크인 날짜가 체크아웃 날짜보다 앞서는지는 두 필드의 관계를 검사하는 별도 규칙이 필요합니다. 스키마는 자료형 설명서이면서 실행 경계이지만, 모든 업무 규칙을 자동으로 추론하는 장치는 아닙니다.